# 01 — Getting Started with `strands-robots`

This notebook covers installation, the lazy-loading import system,
`require_optional()` for dependency management, and path resolution.

In [ ]:
# Install (uncomment the variant you need)
# !pip install strands-robots                    # base: policies + tools
# !pip install "strands-robots[sim-mujoco]"      # + MuJoCo simulation
# !pip install "strands-robots[groot-service]"   # + GR00T ZMQ client
# !pip install "strands-robots[lerobot]"         # + LeRobot local inference (needs GPU)
# !pip install "strands-robots[all]"             # everything

## Lazy Loading

`import strands_robots` is always fast — heavy deps (torch, lerobot, mujoco)
are deferred via `__getattr__` until you actually access them.

The light symbols (`Policy`, `MockPolicy`, `create_policy`) import instantly
because they only depend on stdlib + the base ABC.

In [ ]:
import strands_robots

# These are available instantly — no heavy imports triggered
print(type(strands_robots.Policy))       # <class 'abc.ABCMeta'>
print(type(strands_robots.MockPolicy))   # <class 'type'>
print(type(strands_robots.create_policy))  # <class 'function'>

In [ ]:
# __all__ shows everything — heavy symbols listed but NOT imported yet
print(strands_robots.__all__)

In [ ]:
# Accessing a lazy symbol triggers its import chain:
#   strands_robots.Robot → imports lerobot, torch, numpy
#   strands_robots.Gr00tPolicy → imports numpy
# Uncomment to test:
# Robot = strands_robots.Robot  # slow first time, cached after

## `require_optional()` — Dependency Management

Every optional import in the codebase goes through this single function.
It gives clear error messages with exact install commands.

In [ ]:
from strands_robots.utils import require_optional

# numpy is a base dep — always works
np = require_optional("numpy")
print(f"numpy {np.__version__}")

# zmq is optional — shows helpful error if missing
try:
    zmq = require_optional(
        "zmq",
        pip_install="pyzmq",
        extra="groot-service",
        purpose="GR00T service inference",
    )
    print(f"zmq {zmq.__version__}")
except ImportError as e:
    print(f"ImportError:\n{e}")

## Path Resolution

All user data lives under `~/.strands_robots/` by default.
Override with `STRANDS_ASSETS_DIR` env var.

In [ ]:
from strands_robots.utils import get_base_dir, get_assets_dir, resolve_asset_path

print(f"base_dir:   {get_base_dir()}")       # ~/.strands_robots/
print(f"assets_dir: {get_assets_dir()}")     # ~/.strands_robots/assets/

# resolve_asset_path handles None, relative, and absolute paths
print(f"None+default: {resolve_asset_path(None, 'so100')}")       # .../assets/so100
print(f"relative:     {resolve_asset_path('my_robot')}")          # .../assets/my_robot
print(f"absolute:     {resolve_asset_path('/tmp/model.xml')}")    # /tmp/model.xml
print(f"home:         {resolve_asset_path('~/custom/arm')}")      # /Users/.../custom/arm